# 1. Imports

In [84]:
import numpy as np
import pandas as pd
import os
import time

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import train_test_split

import xgboost as xgb

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cpu


# 2. Dataset Loading

In [69]:
def load_cmapss(train_path, test_path, rul_path):
    cols = ['engine_id', 'cycle'] + \
           [f'op_{i}' for i in range(1, 4)] + \
           [f's_{i}' for i in range(1, 22)]

    # if your files are space-separated, this is correct
    train = pd.read_csv(train_path, sep=r"\s+", header=None)
    test  = pd.read_csv(test_path,  sep=r"\s+", header=None)
    rul   = pd.read_csv(rul_path,   sep=r"\s+", header=None)

    train = train.iloc[:, :len(cols)]
    test  = test.iloc[:,  :len(cols)]

    train.columns = cols
    test.columns  = cols
    rul.columns   = ['RUL_end']
    return train, test, rul

TRAIN_FILE = "train_FD001.csv"
TEST_FILE  = "test_FD001.csv"
RUL_FILE   = "RUL_FD001.csv"

train_df, test_df, rul_df = load_cmapss(TRAIN_FILE, TEST_FILE, RUL_FILE)
print("Train:", train_df.shape, "| Test:", test_df.shape, "| RUL:", rul_df.shape)
train_df.head()

Train: (20631, 26) | Test: (13096, 26) | RUL: (100, 1)


,engine_id,cycle,op_1,op_2,op_3,s_1,s_2,s_3,s_4,s_5,...,s_12,s_13,s_14,s_15,s_16,s_17,s_18,s_19,s_20,s_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


# 3. RUL Label Creation

In [70]:
def add_rul_train(df, cap=125):
    df = df.copy()
    max_cycle = df.groupby('engine_id')['cycle'].max()
    df['RUL'] = df['engine_id'].map(max_cycle) - df['cycle']
    df['RUL'] = df['RUL'].clip(upper=cap)
    return df

def add_rul_test(test_df, rul_end_df, cap=125):
    test_df = test_df.copy()
    max_cycle = test_df.groupby('engine_id')['cycle'].max().reset_index()
    max_cycle.columns = ['engine_id', 'max_cycle']
    max_cycle['RUL_end'] = rul_end_df['RUL_end'].values  # assumes correct order
    
    test_df = test_df.merge(max_cycle, on='engine_id', how='left')
    test_df['RUL'] = (test_df['max_cycle'] - test_df['cycle']) + test_df['RUL_end']
    test_df['RUL'] = test_df['RUL'].clip(upper=cap)
    test_df.drop(columns=['max_cycle','RUL_end'], inplace=True)
    return test_df

CAP = 125
train_df = add_rul_train(train_df, cap=CAP)
test_df  = add_rul_test(test_df, rul_df, cap=CAP)

train_df[['engine_id','cycle','RUL']].head()

,engine_id,cycle,RUL
0,1,1,125
1,1,2,125
2,1,3,125
3,1,4,125
4,1,5,125


# 4. Data Splitting and Scaling

In [71]:
def split_by_engine(df, val_size=0.2, random_state=42):
    groups = df['engine_id'].values
    splitter = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=random_state)
    tr_idx, va_idx = next(splitter.split(df, groups=groups))
    return df.iloc[tr_idx].copy(), df.iloc[va_idx].copy()

op_cols = [c for c in train_df.columns if c.startswith("op_")]
s_cols  = [c for c in train_df.columns if c.startswith("s_")]
feature_cols = op_cols + s_cols

tr_df, va_df = split_by_engine(train_df, val_size=0.2, random_state=42)

scaler = StandardScaler()
scaler.fit(tr_df[feature_cols].values)

def apply_scaler(df):
    df = df.copy()
    df[feature_cols] = scaler.transform(df[feature_cols].values)
    return df

tr_df = apply_scaler(tr_df)
va_df = apply_scaler(va_df)
te_df = apply_scaler(test_df)

print("Train engines:", tr_df.engine_id.nunique(), " Val engines:", va_df.engine_id.nunique())

Train engines: 80  Val engines: 20


# 5. Model Definitions

In [72]:
class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class BiLSTMReg(nn.Module):
    def __init__(self, n_features, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features, hidden_size=hidden,
            num_layers=layers, batch_first=True,
            bidirectional=True,
            dropout=dropout if layers > 1 else 0.0
        )
        self.head = nn.Sequential(
            nn.Linear(hidden*2, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1)
        )
    def forward(self, x):
        out, _ = self.lstm(x)      # (B,T,2H)
        h_last = out[:, -1, :]     # (B,2H)
        return self.head(h_last)

class GRUAttnReg(nn.Module):
    def __init__(self, n_features, hidden=64, layers=1, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=n_features, hidden_size=hidden,
            num_layers=layers, batch_first=True,
            dropout=dropout if layers > 1 else 0.0
        )
        self.attn = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1)
        )
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1)
        )
    def forward(self, x):
        h, _ = self.gru(x)                 # (B,T,H)
        scores = self.attn(h)              # (B,T,1)
        w = torch.softmax(scores, dim=1)   # (B,T,1)
        ctx = (w * h).sum(dim=1)           # (B,H)
        return self.head(ctx)

# 6. Training and Prediction Functions

In [73]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def train_torch(model, train_loader, val_loader, epochs=20, lr=1e-3, device=DEVICE):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.HuberLoss(delta=10.0)

    best_val = float("inf")
    best_state = None

    for ep in range(1, epochs+1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        # val
        model.eval()
        ys, ps = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                pv = model(xb).cpu().numpy().ravel()
                ys.append(yb.numpy().ravel())
                ps.append(pv)
        yv = np.concatenate(ys); pv = np.concatenate(ps)
        val_mae = mean_absolute_error(yv, pv)

        if val_mae < best_val:
            best_val = val_mae
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(f"Epoch {ep:02d} | Val MAE={val_mae:.3f} | Val RMSE={rmse(yv,pv):.3f}")

    model.load_state_dict(best_state)
    model.to(device)
    return model

def predict_torch(model, X, batch=256, device=DEVICE):
    model.eval()
    preds = []
    loader = DataLoader(torch.tensor(X, dtype=torch.float32), batch_size=batch, shuffle=False)
    with torch.no_grad():
        for xb in loader:
            xb = xb.to(device)
            preds.append(model(xb).cpu().numpy().ravel())
    return np.concatenate(preds)

# 7. Sequence Generation

In [74]:
def find_file(name):
    candidates = [
        name,
        name.replace(".csv", ".txt"),
        name.replace(".txt", ".csv"),
        f"/mnt/data/{name}",
        f"/mnt/data/{name.replace('.csv', '.txt')}",
        f"/mnt/data/{name.replace('.txt', '.csv')}"
    ]
    
    for path in candidates:
        if os.path.exists(path):
            return path
    
    raise FileNotFoundError(f"File not found: {name}. Make sure it is uploaded in the same folder.")

def create_sequences(df, feature_cols, window=30, target_col="RUL"):
    X, y = [], []
    
    for engine_id, group in df.groupby("engine_id"):
        group = group.sort_values("cycle")
        features = group[feature_cols].values
        targets = group[target_col].values
        
        if len(group) < window:
            continue
        
        for i in range(window, len(group) + 1):
            X.append(features[i-window:i])
            y.append(targets[i-1])
    
    return np.array(X), np.array(y)

def flatten_last_step(X):
    return X[:, -1, :]

In [75]:
WINDOW = 30

X_tr, y_tr = create_sequences(tr_df, feature_cols, window=WINDOW)
X_va, y_va = create_sequences(va_df, feature_cols, window=WINDOW)
X_te, y_te = create_sequences(te_df, feature_cols, window=WINDOW)

Xtr_tab = flatten_last_step(X_tr)
Xva_tab = flatten_last_step(X_va)
Xte_tab = flatten_last_step(X_te)

print(X_tr.shape, y_tr.shape)
print(X_va.shape, y_va.shape)
print(X_te.shape, y_te.shape)

(14241, 30, 24) (14241,)
(3490, 30, 24) (3490,)
(10196, 30, 24) (10196,)


# 8. BiLSTM and GRU + Attention Training

In [76]:
BATCH = 128
tr_loader = DataLoader(SeqDataset(X_tr, y_tr), batch_size=BATCH, shuffle=True)
va_loader = DataLoader(SeqDataset(X_va, y_va), batch_size=BATCH, shuffle=False)

bilstm = BiLSTMReg(n_features=X_tr.shape[-1], hidden=64, layers=2, dropout=0.2)
bilstm = train_torch(bilstm, tr_loader, va_loader, epochs=20, lr=1e-3)

gru_attn = GRUAttnReg(n_features=X_tr.shape[-1], hidden=64, layers=1, dropout=0.2)
gru_attn = train_torch(gru_attn, tr_loader, va_loader, epochs=20, lr=1e-3)

pred_bi = predict_torch(bilstm, X_va)
pred_gr = predict_torch(gru_attn, X_va)

print("VAL BiLSTM  MAE/RMSE:", mean_absolute_error(y_va,pred_bi), rmse(y_va,pred_bi))
print("VAL GRUAttn MAE/RMSE:", mean_absolute_error(y_va,pred_gr), rmse(y_va,pred_gr))

Epoch 01 | Val MAE=37.026 | Val RMSE=45.080
Epoch 02 | Val MAE=12.160 | Val RMSE=16.046
Epoch 03 | Val MAE=9.415 | Val RMSE=13.200
Epoch 04 | Val MAE=9.089 | Val RMSE=12.742
Epoch 05 | Val MAE=9.526 | Val RMSE=13.464
Epoch 06 | Val MAE=9.081 | Val RMSE=12.463
Epoch 07 | Val MAE=8.770 | Val RMSE=12.592
Epoch 08 | Val MAE=9.244 | Val RMSE=12.847
Epoch 09 | Val MAE=9.386 | Val RMSE=12.654
Epoch 10 | Val MAE=8.836 | Val RMSE=12.548
Epoch 11 | Val MAE=9.694 | Val RMSE=13.637
Epoch 12 | Val MAE=8.676 | Val RMSE=12.595
Epoch 13 | Val MAE=9.823 | Val RMSE=13.821
Epoch 14 | Val MAE=8.811 | Val RMSE=12.944
Epoch 15 | Val MAE=8.698 | Val RMSE=12.566
Epoch 16 | Val MAE=9.867 | Val RMSE=13.467
Epoch 17 | Val MAE=9.134 | Val RMSE=13.355
Epoch 18 | Val MAE=9.874 | Val RMSE=14.013
Epoch 19 | Val MAE=9.432 | Val RMSE=13.771
Epoch 20 | Val MAE=9.641 | Val RMSE=13.946
Epoch 01 | Val MAE=42.467 | Val RMSE=51.411
Epoch 02 | Val MAE=13.917 | Val RMSE=17.911
Epoch 03 | Val MAE=12.313 | Val RMSE=16.842
Epoch 

# 9. XGBoost Model

In [78]:
def train_xgb_native(X_train, y_train, X_val, y_val):
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval   = xgb.DMatrix(X_val,   label=y_val)

    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "eta": 0.03,
        "max_depth": 6,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "lambda": 1.0,
        "seed": 42,
    }

    booster = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=5000,
        evals=[(dtrain, "train"), (dval, "val")],
        early_stopping_rounds=100,
        verbose_eval=200
    )
    return booster

xgb_model = train_xgb_native(Xtr_tab, y_tr, Xva_tab, y_va)
pred_xg = xgb_model.predict(xgb.DMatrix(Xva_tab))

print("VAL XGBoost MAE/RMSE:", mean_absolute_error(y_va,pred_xg), rmse(y_va,pred_xg))

[0]	train-rmse:40.86697	val-rmse:40.79454
[200]	train-rmse:16.42254	val-rmse:17.32613
[280]	train-rmse:15.67380	val-rmse:17.36685
VAL XGBoost MAE/RMSE: 12.831038070271555 17.36684928380196


# 10. Ensemble Validation Results

In [79]:
ens_val = (pred_bi + pred_gr + pred_xg) / 3.0

results = pd.DataFrame({
    "Model": ["BiLSTM", "GRU+Attention", "XGBoost (native)", "Ensemble(avg)"],
    "MAE": [
        mean_absolute_error(y_va, pred_bi),
        mean_absolute_error(y_va, pred_gr),
        mean_absolute_error(y_va, pred_xg),
        mean_absolute_error(y_va, ens_val),
    ],
    "RMSE": [
        rmse(y_va, pred_bi),
        rmse(y_va, pred_gr),
        rmse(y_va, pred_xg),
        rmse(y_va, ens_val),
    ]
}).sort_values("MAE")

results

,Model,MAE,RMSE
0,BiLSTM,8.676478,12.595207
1,GRU+Attention,8.793225,12.672503
3,Ensemble(avg),8.830998,12.121388
2,XGBoost (native),12.831038,17.366849


# 11. Final Test Results

In [80]:
pred_bi_te = predict_torch(bilstm, X_te)
pred_gr_te = predict_torch(gru_attn, X_te)
pred_xg_te = xgb_model.predict(xgb.DMatrix(Xte_tab))
ens_te = (pred_bi_te + pred_gr_te + pred_xg_te) / 3.0

print("TEST Ensemble MAE/RMSE:", mean_absolute_error(y_te, ens_te), rmse(y_te, ens_te))

TEST Ensemble MAE/RMSE: 10.402570511276183 14.201322513487522


# 12. FD002–FD004 Helper Functions

In [81]:
def run_subset_experiment(subset, cap=125, window=30, epochs=20, batch_size=128):
    print("=" * 70)
    print(f"Running experiment for {subset}")
    print("=" * 70)
    
    start_time = time.time()
    
    train_path = find_file(f"train_{subset}.csv")
    test_path = find_file(f"test_{subset}.csv")
    rul_path = find_file(f"RUL_{subset}.csv")
    
    train_df, test_df, rul_df = load_cmapss(train_path, test_path, rul_path)
    
    train_df = add_rul_train(train_df, cap=cap)
    test_df = add_rul_test(test_df, rul_df, cap=cap)
    
    op_cols = [c for c in train_df.columns if c.startswith("op_")]
    s_cols = [c for c in train_df.columns if c.startswith("s_")]
    feature_cols = op_cols + s_cols
    
    tr_df, va_df = split_by_engine(train_df, val_size=0.2, random_state=42)
    
    scaler = StandardScaler()
    scaler.fit(tr_df[feature_cols].values)
    
    def scale_df(df):
        df = df.copy()
        df[feature_cols] = scaler.transform(df[feature_cols].values)
        return df
    
    tr_df = scale_df(tr_df)
    va_df = scale_df(va_df)
    te_df = scale_df(test_df)
    
    X_tr, y_tr = create_sequences(tr_df, feature_cols, window=window)
    X_va, y_va = create_sequences(va_df, feature_cols, window=window)
    X_te, y_te = create_sequences(te_df, feature_cols, window=window)
    
    Xtr_tab = flatten_last_step(X_tr)
    Xva_tab = flatten_last_step(X_va)
    Xte_tab = flatten_last_step(X_te)
    
    print("Train sequences:", X_tr.shape)
    print("Validation sequences:", X_va.shape)
    print("Test sequences:", X_te.shape)
    
    tr_loader = DataLoader(SeqDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    va_loader = DataLoader(SeqDataset(X_va, y_va), batch_size=batch_size, shuffle=False)
    
    set_seed(42)
    bilstm_model = BiLSTMReg(n_features=X_tr.shape[-1], hidden=64, layers=2, dropout=0.2)
    bilstm_model = train_torch(bilstm_model, tr_loader, va_loader, epochs=epochs, lr=1e-3)
    
    set_seed(42)
    gru_model = GRUAttnReg(n_features=X_tr.shape[-1], hidden=64, layers=1, dropout=0.2)
    gru_model = train_torch(gru_model, tr_loader, va_loader, epochs=epochs, lr=1e-3)
    
    xgb_model = train_xgb_native(Xtr_tab, y_tr, Xva_tab, y_va)
    
    pred_bi_va = predict_torch(bilstm_model, X_va)
    pred_gr_va = predict_torch(gru_model, X_va)
    pred_xg_va = xgb_model.predict(xgb.DMatrix(Xva_tab))
    pred_ens_va = (pred_bi_va + pred_gr_va + pred_xg_va) / 3.0
    
    pred_bi_te = predict_torch(bilstm_model, X_te)
    pred_gr_te = predict_torch(gru_model, X_te)
    pred_xg_te = xgb_model.predict(xgb.DMatrix(Xte_tab))
    pred_ens_te = (pred_bi_te + pred_gr_te + pred_xg_te) / 3.0
    
    subset_results = pd.DataFrame({
        "Subset": [subset] * 8,
        "Split": ["Validation", "Validation", "Validation", "Validation",
                  "Test", "Test", "Test", "Test"],
        "Model": ["BiLSTM", "GRU+Attention", "XGBoost", "Ensemble(avg)",
                  "BiLSTM", "GRU+Attention", "XGBoost", "Ensemble(avg)"],
        "MAE": [
            mean_absolute_error(y_va, pred_bi_va),
            mean_absolute_error(y_va, pred_gr_va),
            mean_absolute_error(y_va, pred_xg_va),
            mean_absolute_error(y_va, pred_ens_va),
            mean_absolute_error(y_te, pred_bi_te),
            mean_absolute_error(y_te, pred_gr_te),
            mean_absolute_error(y_te, pred_xg_te),
            mean_absolute_error(y_te, pred_ens_te)
        ],
        "RMSE": [
            rmse(y_va, pred_bi_va),
            rmse(y_va, pred_gr_va),
            rmse(y_va, pred_xg_va),
            rmse(y_va, pred_ens_va),
            rmse(y_te, pred_bi_te),
            rmse(y_te, pred_gr_te),
            rmse(y_te, pred_xg_te),
            rmse(y_te, pred_ens_te)
        ]
    })
    
    elapsed = (time.time() - start_time) / 60
    print(f"Finished {subset} in {elapsed:.2f} minutes")
    
    return subset_results

In [82]:
def load_cmapss(train_path, test_path, rul_path):
    
    cols = (
        ["engine_id", "cycle"] +
        [f"op_{i}" for i in range(1, 4)] +
        [f"s_{i}" for i in range(1, 22)]
    )
    
    train_df = pd.read_csv(
        train_path,
        sep=r"\s+",
        header=None
    )
    
    test_df = pd.read_csv(
        test_path,
        sep=r"\s+",
        header=None
    )
    
    rul_df = pd.read_csv(
        rul_path,
        sep=r"\s+",
        header=None
    )
    
    train_df = train_df.iloc[:, :26]
    test_df = test_df.iloc[:, :26]
    
    train_df.columns = cols
    test_df.columns = cols
    
    rul_df.columns = ["RUL"]
    
    return train_df, test_df, rul_df


def add_rul_train(df, cap=125):
    
    max_cycle = (
        df.groupby("engine_id")["cycle"]
        .max()
        .reset_index()
    )
    
    max_cycle.columns = ["engine_id", "max_cycle"]
    
    df = df.merge(max_cycle, on="engine_id")
    
    df["RUL"] = df["max_cycle"] - df["cycle"]
    
    df["RUL"] = df["RUL"].clip(upper=cap)
    
    df.drop(columns=["max_cycle"], inplace=True)
    
    return df


def add_rul_test(test_df, rul_df, cap=125):
    
    max_cycle = (
        test_df.groupby("engine_id")["cycle"]
        .max()
        .reset_index()
    )
    
    max_cycle.columns = ["engine_id", "max_cycle"]
    
    rul_df["engine_id"] = rul_df.index + 1
    
    max_cycle = max_cycle.merge(rul_df, on="engine_id")
    
    max_cycle["final_cycle"] = (
        max_cycle["max_cycle"] + max_cycle["RUL"]
    )
    
    test_df = test_df.merge(
        max_cycle[["engine_id", "final_cycle"]],
        on="engine_id"
    )
    
    test_df["RUL"] = (
        test_df["final_cycle"] - test_df["cycle"]
    )
    
    test_df["RUL"] = test_df["RUL"].clip(upper=cap)
    
    test_df.drop(columns=["final_cycle"], inplace=True)
    
    return test_df


def split_by_engine(df, val_size=0.2, random_state=42):
    
    engine_ids = df["engine_id"].unique()
    
    train_ids, val_ids = train_test_split(
        engine_ids,
        test_size=val_size,
        random_state=random_state
    )
    
    train_df = df[df["engine_id"].isin(train_ids)].copy()
    val_df = df[df["engine_id"].isin(val_ids)].copy()
    
    return train_df, val_df

# 13. FD002–FD004 Experiments

In [85]:
additional_subsets = ["FD002", "FD003", "FD004"]

all_additional_results = []

for subset in additional_subsets:
    result = run_subset_experiment(
        subset=subset,
        cap=125,
        window=30,
        epochs=20,
        batch_size=128
    )
    all_additional_results.append(result)

additional_results_df = pd.concat(all_additional_results, ignore_index=True)
additional_results_df

Running experiment for FD002
Train sequences: (37432, 30, 24)
Validation sequences: (8787, 30, 24)
Test sequences: (26505, 30, 24)
Epoch 01 | Val MAE=36.910 | Val RMSE=43.109


KeyboardInterrupt: 

In [86]:
best_models = (
    additional_results_df[
        additional_results_df["Split"] == "Test"
    ]
    .sort_values("RMSE")
    .groupby("Subset")
    .first()
    .reset_index()
    .sort_values("RMSE")
)

best_models

,Subset,Split,Model,MAE,RMSE
1,FD003,Test,GRU+Attention,7.183408,11.850962
2,FD004,Test,Ensemble(avg),10.509458,17.437312
0,FD002,Test,Ensemble(avg),14.363186,18.616585


# 14. Group K-Fold Cross Validation

In [ ]:
def run_group_kfold_subset(subset, cap=125, window=30, k=5, epochs=10, batch_size=128):
    
    train_path = find_file(f"train_{subset}.csv")
    test_path = find_file(f"test_{subset}.csv")
    rul_path = find_file(f"RUL_{subset}.csv")
    
    train_df, test_df, rul_df = load_cmapss(train_path, test_path, rul_path)
    train_df = add_rul_train(train_df, cap=cap)
    
    op_cols = [c for c in train_df.columns if c.startswith("op_")]
    s_cols = [c for c in train_df.columns if c.startswith("s_")]
    feature_cols = op_cols + s_cols
    
    engine_ids = train_df["engine_id"].unique()
    groups = engine_ids
    
    gkf = GroupKFold(n_splits=k)
    fold_results = []
    
    for fold, (train_idx, val_idx) in enumerate(gkf.split(engine_ids, groups=groups), start=1):
        print("=" * 60)
        print(f"{subset} | Fold {fold}/{k}")
        print("=" * 60)
        
        train_engines = engine_ids[train_idx]
        val_engines = engine_ids[val_idx]
        
        tr_df = train_df[train_df["engine_id"].isin(train_engines)].copy()
        va_df = train_df[train_df["engine_id"].isin(val_engines)].copy()
        
        scaler = StandardScaler()
        scaler.fit(tr_df[feature_cols].values)
        
        tr_df[feature_cols] = scaler.transform(tr_df[feature_cols].values)
        va_df[feature_cols] = scaler.transform(va_df[feature_cols].values)
        
        X_tr, y_tr = create_sequences(tr_df, feature_cols, window=window)
        X_va, y_va = create_sequences(va_df, feature_cols, window=window)
        
        Xtr_tab = flatten_last_step(X_tr)
        Xva_tab = flatten_last_step(X_va)
        
        tr_loader = DataLoader(SeqDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
        va_loader = DataLoader(SeqDataset(X_va, y_va), batch_size=batch_size, shuffle=False)
        
        set_seed(42)
        bilstm_model = BiLSTMReg(n_features=X_tr.shape[-1], hidden=64, layers=2, dropout=0.2)
        bilstm_model = train_torch(bilstm_model, tr_loader, va_loader, epochs=epochs, lr=1e-3)
        
        set_seed(42)
        gru_model = GRUAttnReg(n_features=X_tr.shape[-1], hidden=64, layers=1, dropout=0.2)
        gru_model = train_torch(gru_model, tr_loader, va_loader, epochs=epochs, lr=1e-3)
        
        xgb_model = train_xgb_native(Xtr_tab, y_tr, Xva_tab, y_va)
        
        pred_bilstm = predict_torch(bilstm_model, X_va)
        pred_gru = predict_torch(gru_model, X_va)
        pred_xgb = xgb_model.predict(xgb.DMatrix(Xva_tab))
        pred_ensemble = (pred_bilstm + pred_gru + pred_xgb) / 3
        
        fold_results.append(["BiLSTM", fold, mean_absolute_error(y_va, pred_bilstm), rmse(y_va, pred_bilstm)])
        fold_results.append(["GRU+Attention", fold, mean_absolute_error(y_va, pred_gru), rmse(y_va, pred_gru)])
        fold_results.append(["XGBoost", fold, mean_absolute_error(y_va, pred_xgb), rmse(y_va, pred_xgb)])
        fold_results.append(["Ensemble(avg)", fold, mean_absolute_error(y_va, pred_ensemble), rmse(y_va, pred_ensemble)])
    
    results_df = pd.DataFrame(
        fold_results,
        columns=["Model", "Fold", "MAE", "RMSE"]
    )
    
    results_df.insert(0, "Subset", subset)
    
    return results_df

In [ ]:
kfold_fd001 = run_group_kfold_subset(
    subset="FD001",
    cap=125,
    window=30,
    k=5,
    epochs=10,
    batch_size=128
)

kfold_fd001

In [ ]:
kfold_summary = (
    kfold_fd001
    .groupby(["Subset", "Model"])
    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std")
    )
    .reset_index()
    .sort_values("RMSE_mean")
)

kfold_summary

In [ ]:
all_kfold_results = []

for subset in ["FD002", "FD003", "FD004"]:
    result = run_group_kfold_subset(
        subset=subset,
        cap=125,
        window=30,
        k=5,
        epochs=10,
        batch_size=128
    )
    all_kfold_results.append(result)

all_kfold_results_df = pd.concat(all_kfold_results, ignore_index=True)

all_kfold_summary = (
    all_kfold_results_df
    .groupby(["Subset", "Model"])
    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std")
    )
    .reset_index()
    .sort_values(["Subset", "RMSE_mean"])
)

all_kfold_summary

# 15. Final Comparison Tables

In [ ]:
final_all_results = all_kfold_summary.copy()

final_all_results = final_all_results.round({
    "MAE_mean": 2,
    "MAE_std": 2,
    "RMSE_mean": 2,
    "RMSE_std": 2
})

final_all_results

In [ ]:
best_model_per_subset = (
    all_kfold_summary
    .sort_values(["Subset", "RMSE_mean"])
    .groupby("Subset")
    .first()
    .reset_index()
)

best_model_per_subset = best_model_per_subset.round({
    "MAE_mean": 2,
    "MAE_std": 2,
    "RMSE_mean": 2,
    "RMSE_std": 2
})

best_model_per_subset

# 16. Conclusion

This notebook presents the full RUL prediction workflow using BiLSTM, GRU with Attention, XGBoost, Ensemble Learning, FD001–FD004 experiments, and Group K-Fold Cross Validation. All models, results, metrics, and experiment settings were preserved exactly.